# NTILC Full Inference Notebook

This notebook wires together:
- cluster retrieval checkpoint
- cluster ID -> tool mapper (software layer)
- dispatcher (dry-run or execution)
- Qwen + LoRA command generation


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(project_root))

project_root


PosixPath('/scratch4/home/akrik/NTILC')

In [2]:
from inference import NTILCOrchestratorAgent
from models.software_layer import ClusterToolMapper, ToolDispatcher


/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
INTENT_CKPT = project_root / 'checkpoints/intent_embedder/best_model.pt'
RETRIEVAL_CKPT = project_root / 'checkpoints/cluster_retrieval/best_model.pt'
LORA_ADAPTER = project_root / 'checkpoints/lora_nl_command_full'
BASE_MODEL = 'Qwen/Qwen3.5-9B'

INTENT_CKPT, RETRIEVAL_CKPT, LORA_ADAPTER


(PosixPath('/scratch4/home/akrik/NTILC/checkpoints/intent_embedder/best_model.pt'),
 PosixPath('/scratch4/home/akrik/NTILC/checkpoints/cluster_retrieval/best_model.pt'),
 PosixPath('/scratch4/home/akrik/NTILC/checkpoints/lora_nl_command_full'))

In [4]:
agent = NTILCOrchestratorAgent.from_pretrained(
    intent_embedder_path=str(INTENT_CKPT),
    query_encoder_path=str(RETRIEVAL_CKPT),
    qwen_model_name_or_path=BASE_MODEL,
    lora_adapter_path=str(LORA_ADAPTER),
    lora_mode='full',
    auto_register_shell_tools=True,
    tool_timeout_seconds=20,
    tool_cwd=str(project_root),
)

agent


`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 759/759 [00:00<00:00, 1475.01it/s, Materializing param=visual.pos_embed.weight]                                 
Qwen3_5Model LOAD REPORT from: Qwen/Qwen3.5-9B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 427/427 [00:02<00:00, 144.23it/s, Materializing param=model.norm.weight]                              


In [5]:
request = 'list all files in the current directory and read the content of README.md'

run = agent.run(
    request=request,
    execute_tools=False,  # dry-run first
    top_k_candidates=3,
    max_retries=2,
)

run.to_dict()


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{'request': 'list all files in the current directory and read the content of README.md',
 'plan_block': '<plan>\n  <action><len:73>list all files in the current directory and read the content of README.md</len></action>\n</plan>',
 'success': True,
 'candidates': [{'cluster_id': 221, 'score': 0.6484375, 'tool_name': 'du'},
  {'cluster_id': 1019, 'score': 0.64453125, 'tool_name': 'pmdasimple'},
  {'cluster_id': 689, 'score': 0.63671875, 'tool_name': 'mariadb-admin'}],
 'steps': [{'candidate': {'cluster_id': 221,
    'score': 0.6484375,
    'tool_name': 'du'},
   'generated_text': "sh -c 'ls; cat README.md'",
   'command': 'du -c ls; cat README.md',
   'dispatch_arguments': {'command': 'du -c ls; cat README.md',
    'query': 'list all files in the current directory and read the content of README.md'},
   'dispatch_result': {'ok': True,
    'tool': 'du',
    'cluster_id': 221,
    'arguments': {'command': 'du -c ls; cat README.md',
     'query': 'list all files in the current directory an

In [6]:
print(run.plan_block)
for i, step in enumerate(run.steps, start=1):
    print(f'\n=== STEP {i} ===')
    print('Candidate:', step.candidate.to_dict())
    print('Generated:', step.generated_text)
    print('Command:', step.command)
    print(step.dispatch_block)
    print(step.response_block)


<plan>
  <action><len:73>list all files in the current directory and read the content of README.md</len></action>
</plan>

=== STEP 1 ===
Candidate: {'cluster_id': 221, 'score': 0.6484375, 'tool_name': 'du'}
Generated: sh -c 'ls; cat README.md'
Command: du -c ls; cat README.md
<dispatch>
  <tool><len:2>du</len></tool>
  <arg name="command"><len:23>du -c ls; cat README.md</len></arg>
  <arg name="query"><len:73>list all files in the current directory and read the content of README.md</len></arg>
</dispatch>
<response>
  <tool><len:2>du</len></tool>
  <status><len:2>ok</len></status>
  <text><len:2>ok</len></text>
  <retry><len:5>false</len></retry>
</response>


## Manual Software Layer Access
Use this if you want direct mapper/dispatcher control without running the full agent.


In [7]:
mapper = ClusterToolMapper.from_retrieval_checkpoint(RETRIEVAL_CKPT)
mapper.register_shell_tools_for_all_clusters(timeout_seconds=20, cwd=project_root)
dispatcher = ToolDispatcher(mapper=mapper, fail_on_nonzero_exit=True)

# Example: map cluster 0 to its tool and dry-run dispatch
tool_for_cluster_0 = mapper.cluster_to_tool(0)
dry = dispatcher.dispatch_cluster(
    cluster_id=0,
    arguments={'command': f'{tool_for_cluster_0} --help'},
    execute=False,
)

tool_for_cluster_0, dry.to_dict()


('PCPCompat',
 {'ok': True,
  'tool': 'PCPCompat',
  'cluster_id': 0,
  'arguments': {'command': 'PCPCompat --help'},
  'result': None,
  'errors': [],
  'executed': False})

## Optional: Execute Tools
Set `execute_tools=True` only after validating dry-run output.


In [8]:
# exec_run = agent.run(
#     request='list files in the current directory',
#     execute_tools=True,
#     top_k_candidates=3,
#     max_retries=2,
# )
# exec_run.to_dict()
